# Week 5: Foundation Fix Verification

This notebook verifies that the foundation gap has been addressed:
1. Image markers are extracted before stripping
2. Section headers are parsed and populated
3. Metadata is properly populated in chunks
4. Vector store works with new metadata

In [1]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PDF_DIR = Path('../data/raw_pdfs')
CHROMA_DIR = Path('../data/chroma_db_c3')

print(f"PDFs: {list(PDF_DIR.glob('*.pdf'))}")

PDFs: [PosixPath('../data/raw_pdfs/waterpurifier_complex.pdf'), PosixPath('../data/raw_pdfs/airpurifier_simple.pdf'), PosixPath('../data/raw_pdfs/airpurifier_complex_MFL69726859_00_190321_00.pdf'), PosixPath('../data/raw_pdfs/vaccumcleaner_complex.pdf'), PosixPath('../data/raw_pdfs/waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf'), PosixPath('../data/raw_pdfs/vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf')]


## 1. Test Image Marker Extraction

In [2]:
from src.parsing import extract_image_markers, extract_sections, parse_pdf

# Test on sample PDF
sample_pdf = list(PDF_DIR.glob('waterpurifier_complex.pdf'))[0]
parsed = parse_pdf(sample_pdf)

print(f"Source: {parsed.source}")
print(f"Category: {parsed.category}")
print(f"Complexity: {parsed.complexity}")
print(f"\nSections found ({len(parsed.sections)}):")
for s in parsed.sections[:10]:
    print(f"  - {s}")
if len(parsed.sections) > 10:
    print(f"  ... and {len(parsed.sections) - 10} more")

Source: waterpurifier_complex.pdf
Category: waterpurifier
Complexity: complex

Sections found (103):
  - 제품 사용설명서 데스크 정수기
  - 권장 안전 사용 기간 : 7년
  - 전문 기술이 필요한 제품을 설치할 때는 LG전자 서비스 센터를 이용하세요.
  - 안전을 위해 주의하기
  - LG ThinQ 사용하기
  - 사용하기
  - 관리하기
  - 설치하기
  - 고장 신고 전 확인하기
  - 제품 보증서 보기
  ... and 93 more


In [3]:
# Check image markers
page = parsed.pages[0]
print(f"Image markers found: {len(page.image_markers)}")
print(f"\nFirst 5 image markers:")
for img in page.image_markers[:5]:
    print(f"  Position {img['position']}: {img['width']}x{img['height']}")

Image markers found: 33

First 5 image markers:
  Position 0: 97x44
  Position 560: 218x56
  Position 2227: 171x73
  Position 2426: 171x77
  Position 2747: 171x77


## 2. Test Chunking with Metadata

In [4]:
from src.chunking import chunk_pdf, chunks_to_langchain_docs

# Chunk the sample PDF
chunks = chunk_pdf(sample_pdf, chunk_size=1000, chunk_overlap=200)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunks with sections: {sum(1 for c in chunks if c.section)}")
print(f"Chunks with images: {sum(1 for c in chunks if c.image_count > 0)}")
print(f"Total images across chunks: {sum(c.image_count for c in chunks)}")

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 49

Chunks with sections: 49
Chunks with images: 26
Total images across chunks: 42


In [5]:
# Show sample chunks with metadata
print("Sample chunks with metadata:")
print("=" * 70)

for chunk in chunks[:5]:
    print(f"\n[{chunk.chunk_id}]")
    print(f"  Section: {chunk.section}")
    print(f"  Images: {chunk.image_count}")
    print(f"  Text preview: {chunk.text[:100]}...")

Sample chunks with metadata:

[waterpurifier_complex_p001_c000]
  Section: 제품 사용설명서 데스크 정수기
  Images: 1
  Text preview: ## 제품 사용설명서 데스크 정수기 

제품을 안전하고 편리하게 사용하기 위해 반드시 제품을 사용하기 전에 사용설명서를 읽어주세요. 제품 보증서도 함께 들어있으니 잘 보관하세요. ...

[waterpurifier_complex_p001_c001]
  Section: 관리하기
  Images: 0
  Text preview: - 27 정수 필터 교체하기 

## 설치하기 

- 30 제품을 설치할 때 

## 고장 신고 전 확인하기 

- 31 고장 진단하기 32 문제 해결하기 

## 제품 보증서 보...

[waterpurifier_complex_p001_c002]
  Section: 제품을 설치할 때
  Images: 3
  Text preview: - 물 또는 빗물이 튀는 곳이나 습기가 많은 곳에 설치하지 마세요. 

- 먼지가 많거나 온도 변화가 심한 장소(실외, 비닐하우스 등)에 설치하지 마세요. 

- LG전자 서비스 ...

[waterpurifier_complex_p001_c003]
  Section: 전원 플러그나 전원선을 다룰 때
  Images: 2
  Text preview: - 제품 뒷면에 여러 개의 휴대용 콘센트 또는 휴대용 전원 공급 장치를 두지 마세요. 전원 플러그를 제품 뒷면에 눌리게 하거나 거꾸로 꽂지 마세요. 

- 전원 플러그를 꽂았다 뺏...

[waterpurifier_complex_p001_c004]
  Section: 제품을 사용할 때
  Images: 1
  Text preview: ## 제품을 사용할 때 

## R 금지 사항 

- 제품에 매달리거나 제품 위에 올라가지 마세요. 특히 어린이는 조심시켜 주세요. 

- 제품의 냉기가 나오는 곳, 덮개 부분, ...


In [6]:
# Find chunks with images and show their sections
print("Chunks with images:")
print("=" * 70)

image_chunks = [c for c in chunks if c.image_count > 0]
for chunk in image_chunks[:10]:
    print(f"\n[{chunk.chunk_id}] Section: {chunk.section}")
    print(f"  Images: {chunk.image_count}")
    for img in chunk.image_markers:
        print(f"    - {img['width']}x{img['height']} at pos {img['position']}")

Chunks with images:

[waterpurifier_complex_p001_c000] Section: 제품 사용설명서 데스크 정수기
  Images: 1
    - 218x56 at pos 560

[waterpurifier_complex_p001_c002] Section: 제품을 설치할 때
  Images: 3
    - 171x73 at pos 2227
    - 171x77 at pos 2426
    - 171x77 at pos 2747

[waterpurifier_complex_p001_c003] Section: 전원 플러그나 전원선을 다룰 때
  Images: 2
    - 171x77 at pos 2747
    - 171x80 at pos 3643

[waterpurifier_complex_p001_c004] Section: 제품을 사용할 때
  Images: 1
    - 171x80 at pos 3643

[waterpurifier_complex_p001_c005] Section: 제품을 사용할 때
  Images: 1
    - 50x44 at pos 4943

[waterpurifier_complex_p001_c007] Section: 제품을 사용할 때
  Images: 1
    - 172x49 at pos 7140

[waterpurifier_complex_p001_c008] Section: 제품을 재등록하거나 사용자를 추가 등록하는 경우
  Images: 1
    - 172x49 at pos 7140

[waterpurifier_complex_p001_c010] Section: 기본 출수 변경
  Images: 1
    - 172x48 at pos 9805

[waterpurifier_complex_p001_c011] Section: 방해 금지 모드 설정
  Images: 1
    - 172x48 at pos 9805

[waterpurifier_complex_p001_c015] Section: 온수를 사용할 때
 

## 3. Test LangChain Document Conversion

In [7]:
# Convert to LangChain docs
docs = chunks_to_langchain_docs(chunks)

print(f"Total documents: {len(docs)}")
print(f"\nSample document metadata:")
print(docs[5].metadata)

Total documents: 49

Sample document metadata:
{'chunk_id': 'waterpurifier_complex_p001_c005', 'source': 'waterpurifier_complex.pdf', 'category': 'waterpurifier', 'complexity': 'complex', 'page': 1, 'section': '제품을 사용할 때', 'chunk_index': 5, 'char_count': 942, 'image_count': 1, 'image_ids': 'img_50x44'}


In [8]:
# Verify all metadata fields are populated
sample_meta = docs[5].metadata

required_fields = ['chunk_id', 'source', 'category', 'complexity', 
                   'page', 'section', 'chunk_index', 'char_count', 
                   'image_count', 'image_ids']

print("Metadata field check:")
for field in required_fields:
    value = sample_meta.get(field, 'MISSING')
    status = '✓' if field in sample_meta else '✗'
    print(f"  {status} {field}: {value}")

Metadata field check:
  ✓ chunk_id: waterpurifier_complex_p001_c005
  ✓ source: waterpurifier_complex.pdf
  ✓ category: waterpurifier
  ✓ complexity: complex
  ✓ page: 1
  ✓ section: 제품을 사용할 때
  ✓ chunk_index: 5
  ✓ char_count: 942
  ✓ image_count: 1
  ✓ image_ids: img_50x44


## 4. Create Vector Store with All PDFs

In [9]:
from src.vectorstore import create_vectorstore

# Create vector store with all PDFs
vectorstore, all_chunks = create_vectorstore(
    pdf_dir=PDF_DIR,
    persist_dir=CHROMA_DIR,
    collection_name='lg_manuals_c3',
    chunk_size=1000,
    chunk_overlap=200,
)

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/google/rpc/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Deleted old lg_manuals_c3 collection
Processing airpurifier_complex_MFL69726859_00_190321_00.pdf... 46 chunks, 36 sections, 139 images
Processing airpurifier_simple.pdf... 46 chunks, 35 sections, 146 images
Processing vaccumcleaner_complex.pdf... 58 chunks, 46 sections, 121 images
Processing vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf... 21 chunks, 19 sections, 71 images
Processing waterpurifier_complex.pdf... 49 chunks, 35 sections, 42 images
Processing waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf... 38 chunks, 34 sections, 44 images

Total: 258 chunks


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Created vector store with 258 documents


In [10]:
# Verify metadata in vector store
results = vectorstore._collection.get(limit=5, include=['metadatas'])

print("Sample metadata from vector store:")
for meta in results['metadatas']:
    print(f"\n{meta['chunk_id']}:")
    print(f"  category: {meta['category']}")
    print(f"  section: {meta.get('section', 'N/A')}")
    print(f"  image_count: {meta.get('image_count', 0)}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Sample metadata from vector store:

waterpurifier_complex_p001_c020:
  category: waterpurifier
  section: 제어창 사용하기(WD520A 모델)
  image_count: 1

vacuumcleaner_complex_p001_c000:
  category: vacuumcleaner
  section: 제품 사용설명서 핸디스틱 청소기
  image_count: 2

airpurifier_complex_p001_c021:
  category: airpurifier
  section: 운전 시작하기
  image_count: 3

airpurifier_simple_p001_c018:
  category: airpurifier
  section: 스마트 진단부
  image_count: 3

waterpurifier_complex_p001_c014:
  category: waterpurifier
  section: 일반적으로 사용할 때
  image_count: 0


## 5. Test Retrieval with New Metadata

In [11]:
# Test queries
TEST_QUERIES = [
    {"query": "정수기 필터 교체는 어떻게 하나요?", "expected_category": "waterpurifier"},
    {"query": "공기청정기 필터 청소 방법 알려주세요", "expected_category": "airpurifier"},
    {"query": "청소기 배터리 충전 시간은 얼마나 되나요?", "expected_category": "vacuumcleaner"},
    {"query": "Wi-Fi 연결이 안될 때 어떻게 해야 하나요?", "expected_category": None},
]

retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

for q in TEST_QUERIES:
    docs = retriever.invoke(q['query'])
    print(f"\nQuery: {q['query'][:40]}...")
    print(f"Expected: {q['expected_category']}")
    print(f"Retrieved:")
    for i, doc in enumerate(docs):
        cat = doc.metadata.get('category', '?')
        section = doc.metadata.get('section', 'N/A')
        img_count = doc.metadata.get('image_count', 0)
        match = '✓' if q['expected_category'] is None or cat == q['expected_category'] else '✗'
        print(f"  {match} [{cat}] Section: {section[:30] if section else 'N/A'}... Images: {img_count}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Query: 정수기 필터 교체는 어떻게 하나요?...
Expected: waterpurifier
Retrieved:
  ✓ [waterpurifier] Section: 정수 필터 교체하기... Images: 5
  ✓ [waterpurifier] Section: 정수 필터 교체하기... Images: 3
  ✓ [waterpurifier] Section: LG 정수 필터의 특징... Images: 5
  ✓ [waterpurifier] Section: 정수 필터 교체하기... Images: 2
  ✗ [airpurifier] Section: 상태 표시부 알림... Images: 3

Query: 공기청정기 필터 청소 방법 알려주세요...
Expected: airpurifier
Retrieved:
  ✓ [airpurifier] Section: 청소 및 필터 교체 시 알아두기... Images: 0
  ✓ [airpurifier] Section: 상태 표시부 알림... Images: 3
  ✓ [airpurifier] Section: 상태 표시부 알림... Images: 3
  ✓ [airpurifier] Section: 상태 표시부 알림... Images: 3
  ✗ [waterpurifier] Section: 정수 필터 교체하기... Images: 3

Query: 청소기 배터리 충전 시간은 얼마나 되나요?...
Expected: vacuumcleaner
Retrieved:
  ✓ [vacuumcleaner] Section: 보조 배터리 충전하기... Images: 1
  ✓ [vacuumcleaner] Section: 제품 규격 정보... Images: 1
  ✓ [vacuumcleaner] Section: 오픈 소스 안내 정보... Images: 0
  ✗ [airpurifier] Section: 필터 교체하기... Images: 5
  ✗ [airpurifier] Section: 먼지 농도 확인하기... Images: 2

Query: Wi-Fi 연결

## 6. Section Distribution Analysis

In [12]:
# Analyze section distribution
from collections import Counter

section_counts = Counter(c.section for c in all_chunks if c.section)

print(f"Total unique sections: {len(section_counts)}")
print(f"\nTop 15 sections by chunk count:")
for section, count in section_counts.most_common(15):
    print(f"  {count:3d} chunks: {section}")

Total unique sections: 153

Top 15 sections by chunk count:
   12 chunks: 차  례
   10 chunks: 사용 관련
    7 chunks: 제품을 사용할 때
    6 chunks: 고장 신고 전 확인 사항
    6 chunks: 품질 보증 기간
    6 chunks: 문제 해결하기
    5 chunks: 제품을 설치할 때
    5 chunks: 와이파이 설정하기
    4 chunks: 전원 플러그나 전원선을 다룰 때
    4 chunks: 와이파이
    4 chunks: 유상 서비스 책임 안내
    4 chunks: 정수 필터 교체하기
    3 chunks: 필터 손잡이
    3 chunks: 먼지 농도 확인하기
    3 chunks: 상태 표시부 알림


In [13]:
# Image distribution by section
section_images = {}
for chunk in all_chunks:
    if chunk.section:
        if chunk.section not in section_images:
            section_images[chunk.section] = 0
        section_images[chunk.section] += chunk.image_count

print("Sections with most images:")
sorted_sections = sorted(section_images.items(), key=lambda x: x[1], reverse=True)
for section, count in sorted_sections[:10]:
    print(f"  {count:3d} images: {section}")

Sections with most images:
   18 images: 인화물질을 제품 내부로 투입하지 마십시오.
   18 images: TV나 오디오 가까이 두지 마십시오.
   17 images: 전원 플러그에 물기나 먼지를 완전히 제거한 후 콘센트에 단단히 꽂아 주십시오.
   13 images: 필터 보호 비닐과 고정 테이프 제거하기
   13 images: 정수 필터 교체하기
   12 images: 필터 청소하기
   11 images: 리모컨 사용 준비하기
   11 images: 먼지통 비우기
   10 images: 클린부스터 커버와 그릴이 분리된 상태에서 제품을 운전하지 마십시오.
   10 images: 제품 위에 자석이나 금속물을 놓지 마십시오.


## 7. Summary

**Foundation Gap Status:**
- [ ] Image markers extracted before stripping
- [ ] Section headers parsed from `##` patterns
- [ ] Metadata populated: section, image_count, image_ids
- [ ] Vector store created with new metadata

In [14]:
# Final summary
chunks_with_section = sum(1 for c in all_chunks if c.section)
chunks_with_images = sum(1 for c in all_chunks if c.image_count > 0)
total_images = sum(c.image_count for c in all_chunks)

print("Foundation Gap Fix Summary")
print("=" * 50)
print(f"Total chunks: {len(all_chunks)}")
print(f"Chunks with section: {chunks_with_section} ({chunks_with_section/len(all_chunks)*100:.1f}%)")
print(f"Chunks with images: {chunks_with_images} ({chunks_with_images/len(all_chunks)*100:.1f}%)")
print(f"Total images tracked: {total_images}")
print(f"Unique sections: {len(section_counts)}")
print(f"\n✓ Foundation gap addressed!")

Foundation Gap Fix Summary
Total chunks: 258
Chunks with section: 258 (100.0%)
Chunks with images: 176 (68.2%)
Total images tracked: 563
Unique sections: 153

✓ Foundation gap addressed!
